In [1]:
import pandas as pd
import ast
import re

In [2]:
df = pd.read_csv(r"..\..\Data\clean_data 16-03-2026.csv",parse_dates=['insert_date','first_review_date','last_review_date'])

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7537 entries, 0 to 7536
Data columns (total 36 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   apartment_id                 7537 non-null   int64         
 1   name                         7534 non-null   object        
 2   description                  7537 non-null   object        
 3   host_id                      7537 non-null   int64         
 4   neighbourhood_name           7537 non-null   object        
 5   neighbourhood_district       7537 non-null   object        
 6   room_type                    7537 non-null   object        
 7   accommodates                 7537 non-null   int64         
 8   bathrooms                    7494 non-null   float64       
 9   bedrooms                     7499 non-null   float64       
 10  beds                         7530 non-null   float64       
 11  amenities_list               7537 non-null 

In [ ]:

df['amenities_str'] = df['amenities_list'].astype(str)


# Quitamos llaves, corchetes y comillas
df['amenities_str'] = df['amenities_str'].str.replace(r'[\{\}\[\]\"]', '', regex=True)

# Arreglamos caracteres raros y espacios extraños
df['amenities_str'] = df['amenities_str'].str.replace('u2019', "'", regex=False)
df['amenities_str'] = df['amenities_str'].str.replace('\u00a0', ' ', regex=False)

# Unificamos los errores de escritura 
df['amenities_str'] = df['amenities_str'].str.replace(r"Children.s", "Children's", regex=True)
df['amenities_str'] = df['amenities_str'].str.replace(r"Pack .n Play", "Pack 'n Play", regex=True)
df['amenities_str'] = df['amenities_str'].str.replace('u2013', '-', regex=False)
df['amenities_str'] = df['amenities_str'].str.replace('u00a0', ' ', regex=False)

df['amenities_str'] = df['amenities_str'].str.replace('\u2013', '-', regex=False)


df['amenities_str'] = df['amenities_str'].str.replace('\u00a0', ' ', regex=False)


def extraer_y_filtrar(texto):
    if pd.isna(texto) or texto == 'nan':
        return []
    
    # Separamos por comas
    elementos = texto.split(',')
    lista_final = []
    
    for e in elementos:
        limpio = e.strip()
        # Si no está vacío y NO es un error de traducción o un sin contestar, lo guardamos
        if limpio and "translation missing" not in limpio.lower() and "(sin contestar)" not in limpio.lower():
            lista_final.append(limpio)
            
    return lista_final


# 3. APLICAMOS AL DATAFRAME PRINCIPAL 
df['amenities_final'] = df['amenities_str'].apply(extraer_y_filtrar)



todas_las_amenities = []
for lista in df['amenities_final']:
    todas_las_amenities.extend(lista)

lista_unica = sorted(list(set(todas_las_amenities)))

print(f"Se han encontrado {len(lista_unica)} amenities distintas ")

¡Listo! Se han encontrado 234 amenities distintas y tu DataFrame sigue teniendo 7537 filas.


In [ ]:
cat_baños = {
    'Bath towel': 1, 'Bathroom essentials':1 , 'Bathtub': 4, 'Bidet': 3, 'Body soap': 2,
    'Conditioner': 2, 'En suite bathroom': 10, 'Hair dryer': 4, 'Heated towel rack': 4,
    'Hot water': 1, 'Rituals body soap': 6, 'Rituals shampoo': 6, 'Shampoo': 2,
    'Shower gel': 2, 'Soaking tub': 9, 'Toilet paper': 1, 'Touchless faucets': 6,
    'Walk-in shower': 7, 'toilet': 1
}

cat_cocina_equipada = {
    'Baking sheet': 2, 'Bread maker': 6, 'Breakfast table': 4, 'Coffee maker': 4,
    'Convection oven': 7, 'Cooking basics': 1, 'Dining table': 4, 'Dishes and silverware': 1,
    'Dishwasher': 8, 'Double oven': 9, 'Electric stove': 4, 'Espresso machine': 6,
    'Freezer': 3, 'Full kitchen': 7, 'Gas oven': 4, 'Hot water kettle': 3, 'Kitchen': 8,
    'Kitchenette': 3, 'Microwave': 4, 'Mini fridge': 3, 'Nespresso machine': 6, 'Oven': 5,
    'Pour Over Coffee': 3, 'Pour-over coffee': 3, 'Refrigerator': 3, 'Rice Maker': 4,
    'Stainless steel electric stove': 4, 'Stainless steel oven': 5, 'Stainless steel stove': 4,
    'Steam oven': 4, 'Stove': 2, 'Toaster': 3, 'Warming drawer': 4, 'Wine glasses': 2
}

cat_habitacion = {
    'Bed linens': 1, 'Bedroom comforts': 1, 'Clothing storage': 2, 'Clothing storage: closet': 2,
    'Day bed': 7, 'Extra pillows and blankets': 6, 'Firm mattress': 2, 'Hangers': 2,
    'Memory foam mattress': 7, 'Murphy bed': 1, 'Pillow-top mattress': 9, 'Room-darkening shades': 5
}

cat_lujo = {
    'BBQ grill': 9, 'Balcony': 7, 'Barbecue utensils': 8, 'Beach essentials': 6,
    'Beach view': 10, 'Beachfront': 10, 'Bluetooth sound system': 5, 'Breakfast': 8,
    'Exercise equipment': 10, 'Formal dining area': 8, 'Game console': 8, 'Garden or backyard': 10,
    'Gym': 10, 'Hammock': 7, 'Hot tub': 10, 'IKEA NEARBY Bluetooth sound system': 5,
    'Lake access': 10, 'Mountain view': 8, 'Outdoor dining area': 9, 'Outdoor furniture': 8,
    'Outdoor seating': 8, 'Outdoor shower': 8, 'Patio or balcony': 9, 'Pool': 10,
    'Pool toys': 5, 'Private living room': 6, 'Rain shower': 9, 'Shared garden or backyard': 5,
    'Shared outdoor pool': 7, 'Shared pool': 7, 'Ski in/Ski out': 10, 'Ski-in/Ski-out': 10,
    'Sound system': 5, 'Suitable for events': 9, 'Sun loungers': 6, 'Terrace': 8, 'Waterfront': 10,'Bluetooth sound system': 5
}

cat_seguridad_logistica = {
    '24-hour check-in': 7, 'Building staff': 9, 'Buzzer/wireless intercom': 2,
    'Carbon monoxide alarm': 1, 'Carbon monoxide detector': 1, 'Doorman': 9,
    'Doorman Entry': 9, 'EV charger': 9, 'Fire extinguisher': 1, 'Fireplace guards': 6,
    'First aid kit': 1, 'Free driveway parking on premises - 1 space': 8,
    'Free parking on premises': 7, 'Free parking on street': 6, 'Free street parking': 6,
    'Front desk/doorperson': 9, 'Host greets you': 2, 'Keypad': 8, 'Lock on bedroom door': 8,
    'Lockbox': 8, 'Long term stays allowed': 5, 'Luggage dropoff allowed': 7, 'Mudroom': 7,
    'Paid parking garage off premises': 3, 'Paid parking garage on premises': 3,
    'Paid parking off premises': 3, 'Paid parking on premises': 3, 'Private entrance': 10,
    'Safety card': 6, 'Self Check-In': 8, 'Self check-in': 8, 'Smart lock': 8,
    'Smoke alarm': 1, 'Smoke detector': 1, 'Smoking allowed': 4
}
cat_accesibilidad ={
    'accessible-heightbed': 6
    'accessible-heighttoilet':	4
    'elevator': 4
    'elevatorinbuilding': 4
    'fixedgrabbarsforshower': 4
    'fixedgrabbarsforshower&toilet': 5
    'fixedgrabbarsfortoilet': 4
    'flat': 10
    'flatpathtofrontdoor': 7
    'flatpathtoguestentrance': 8
    'formaldiningarea': 8
    'groundflooraccess': 4
    'mudroom': 8
    'nostairsorstepstoenter': 6
    'roll-inshower': 9
    'showerchair': 3
    'singlelevelhome': 7
    'smoothpathwaytofrontdoor': 8
    'step-freeaccess': 6
    'step-freeshower': 9
    'walk-inshower': 8
    'well-litpathtoentrance': 7
    'wheelchairaccessible': 5
    'wideclearancetobed': 8
    'wideclearancetoshower': 7
    'wideclearancetoshower&toilet': 6
    'widedoorway': 5
    'widedoorwaytoguestbathroom': 5
    'wideentrance': 9
    'wideentranceforguests': 8
    'wideentryway': 9
    'widehallwayclearance': 8
    'widehallways': 8
}
cat_wifi_oficina ={
    'buzzer/wirelessintercom': 6
    'dedicatedworkspace': 8
    'ethernetconnection': 9
    'internet': 3
    'laptop-friendlyworkspace': 5
    'laptopfriendlyworkspace': 6
    'pocketwifi': 6
    'printer': 7
    'wifi': 4
    'wifiu2013100mbps':	8
    'wirelessinternet': 4

}
cat_TV_clima ={
    '40hdtv': 3
    '43hdtvwithnetflix': 6
    'airconditioning': 7
    'amazonecho': 7
    'bluetoothsoundsystem': 8
    'cabletv': 4
    'ceilingfan': 4
    'centralairconditioning': 9
    'centralheating': 8
    'dvdplayer': 1
    'gameconsole': 9
    'hbogo': 7
    'ikeanearbybluetoothsoundsystem': 5
    'netflix': 7
    'portablefans': 2
    'smarttv': 7
    'soundsystem': 9
    'tv': 3

}
cat_family_friendly ={
    'babybath': 4
    'babymonitor': 8
    'babysafetygates': 5
    'babysitterrecommendations': 9
    'changingtable': 7
    'childrenu2019sbooksandtoys': 6
    'childrenu2019sdinnerware': 4
    'children�sbooksandtoys': 6
    'children�sdinnerware': 4
    'crib': 6
    'family/kidfriendly': 3
    'highchair': 5
    'outletcovers': 2
    'packu2019nplay/travelcrib': 5
    'pack�nplay/travelcrib': 7
    'pooltoys': 7
    'stairgates': 5
    'tablecornerguards': 2

}
cat_limpieza_lavanderia ={
    'bathroomessentials': 3
    'bathtowel': 5
    'bathtubwithbathchair': 4
    'cleaningproducts': 2
    'dryer': 9
    'dryingrackforclothing': 3
    'iron': 5
    'laundromatnearby': 1
    'washeru2013u00a0inunit': 8

}